In [0]:
DECLARE OR REPLACE VARIABLE catalog_use = 'main';
DECLARE OR REPLACE VARIABLE schema_use = 'synthea';

In [0]:
SET VARIABLE catalog_use = :catalog_use;
SET VARIABLE schema_use = :schema_use; 

In [0]:
USE IDENTIFIER(catalog_use || '.' || schema_use);
SELECT current_catalog(), current_schema();

In [0]:
FROM identifier(catalog_use || '.information_schema.routines') |>
WHERE specific_schema = schema_use |>
SELECT specific_catalog, specific_schema, specific_name, routine_type, routine_definition, routine_body, data_type;

In [0]:
SELECT
  databricks_rest_get(
    endpoint => '/2.0/sql'
    ,resource => 'warehouses'
    ,path_parameters => NULL
    ,query_parameters => array('run_as_user_id=')
    ,body => NULL
  )

In [0]:
CREATE OR REPLACE FUNCTION databricks_rest_sql_warehouses_list_warehouses(
  run_as_user_id INTEGER COMMENT 'Service Principal which will be used to fetch the list of warehouses. If not specified, the user from the session header is used.' DEFAULT NULL
)
RETURNS VARIANT 
COMMENT 'Performs actions against the Unity Catalog Databricks REST API Endpoint.'
LANGUAGE SQL 
RETURN 
SELECT 
  databricks_rest_get(
    endpoint => '/2.0/sql'
    ,resource => 'warehouses'
    ,path_parameters => NULL
    ,query_parameters => array('run_as_user_id=' || run_as_user_id)
    ,body => NULL
  )
;

In [0]:
WITH response as (
  FROM (
    SELECT 
      databricks_rest_sql_warehouses_list_warehouses() AS listing
  )
  ,LATERAL variant_explode(listing:warehouses) as warehouses |>
  SELECT 
    warehouses.pos as warehouses_pos
    ,warehouses.value as warehouses_value
    ,warehouses.value:id::string as warehouse_id
)
FROM response
,LATERAL variant_explode(warehouses_value) |> 
SELECT warehouse_id, key, value |>
PIVOT (first(value) for key in ("auto_resume","auto_stop_mins","channel","cluster_size","creator_id","creator_name","enable_photon","enable_serverless_compute","health","id","jdbc_url","max_num_clusters","min_num_clusters","name","num_clusters","odbc_params","size","spot_instance_policy","state"))